In [1]:
import sys

print(sys.executable)

C:\Users\logesh kumar\myenv\Scripts\python.exe


In [2]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "scikit-learn"
])

0

In [3]:
import sklearn

print(sklearn.__version__)

1.8.0


In [4]:
from sklearn.model_selection import train_test_split

In [5]:
# ============================================
# 2.1 Import Required Libraries
# ============================================

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Data Manipulation
import pandas as pd
import numpy as np

# Scikit-learn Utilities
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# Save Objects
import joblib
import os

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [6]:
# ============================================
# 2.2 Load Dataset
# ============================================

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

print("Train Shape :", train_df.shape)
print("Test Shape :", test_df.shape)

Train Shape : (140700, 20)
Test Shape : (93800, 19)


In [7]:
# ============================================
# 2.3 Create Backup Copy
# ============================================

train = train_df.copy()
test = test_df.copy()

print("Backup Created Successfully")

Backup Created Successfully


In [8]:
# ============================================
# 2.4 Drop Unnecessary Columns
# ============================================

train.drop(columns=["id", "Name"], inplace=True)
test.drop(columns=["id", "Name"], inplace=True)

print("Unnecessary Columns Removed")

Unnecessary Columns Removed


In [9]:
# ============================================
# 2.5 Separate Numerical and Categorical Columns
# ============================================

# Numerical Columns
numerical_columns = train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

# Categorical Columns
categorical_columns = train.select_dtypes(
    include=["object"]
).columns.tolist()

# Remove Target Column
numerical_columns.remove("Depression")

print("Numerical Columns")
print(numerical_columns)

print("\nCategorical Columns")
print(categorical_columns)

Numerical Columns
['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Work/Study Hours', 'Financial Stress']

Categorical Columns
['Gender', 'City', 'Working Professional or Student', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']


In [10]:
# ============================================
# 2.6 Handle Missing Values
# ============================================

# Numerical Imputer
num_imputer = SimpleImputer(
    strategy="median"
)

# Fit and Transform Train Data
train[numerical_columns] = num_imputer.fit_transform(
    train[numerical_columns]
)

# Transform Test Data
test[numerical_columns] = num_imputer.transform(
    test[numerical_columns]
)

# Categorical Imputer
cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

# Fit and Transform Train Data
train[categorical_columns] = cat_imputer.fit_transform(
    train[categorical_columns]
)

# Transform Test Data
test[categorical_columns] = cat_imputer.transform(
    test[categorical_columns]
)

print("Missing Values Handled Successfully")

Missing Values Handled Successfully


In [11]:
# ============================================
# 2.7 Encode Categorical Features
# ============================================

# Dictionary to Store Encoders
label_encoders = {}

for column in categorical_columns:

    # Convert Values to String
    train[column] = train[column].astype(str)
    test[column] = test[column].astype(str)

    # Create Label Encoder
    le = LabelEncoder()

    # Combine Train and Test Data
    combined_data = pd.concat(
        [train[column], test[column]],
        axis=0
    )

    # Fit Encoder
    le.fit(combined_data)

    # Transform Train Data
    train[column] = le.transform(
        train[column]
    )

    # Transform Test Data
    test[column] = le.transform(
        test[column]
    )

    # Store Encoder
    label_encoders[column] = le

print("Categorical Features Encoded Successfully")

Categorical Features Encoded Successfully


In [12]:
# ============================================
# 2.8 Feature and Target Separation
# ============================================

# Features
X = train.drop(
    "Depression",
    axis=1
)

# Target
y = train["Depression"]

print("Feature Shape :", X.shape)
print("Target Shape :", y.shape)

Feature Shape : (140700, 17)
Target Shape : (140700,)


In [13]:
# ============================================
# 2.9 Train Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y
)

print("X_train Shape :", X_train.shape)
print("X_test Shape :", X_test.shape)

print("y_train Shape :", y_train.shape)
print("y_test Shape :", y_test.shape)

X_train Shape : (112560, 17)
X_test Shape : (28140, 17)
y_train Shape : (112560,)
y_test Shape : (28140,)


In [14]:
# ============================================
# 2.10 Feature Scaling
# ============================================

# Create Scaler
scaler = StandardScaler()

# Fit and Transform Train Data
X_train = scaler.fit_transform(
    X_train
)

# Transform Test Data
X_test = scaler.transform(
    X_test
)

# Transform Submission Test Data
test_scaled = scaler.transform(
    test
)

print("Feature Scaling Completed")

Feature Scaling Completed


In [15]:
# ============================================
# 2.11 Save Preprocessing Objects
# ============================================

# Create Artifacts Folder
os.makedirs(
    "../artifacts",
    exist_ok=True
)

# Save Scaler
joblib.dump(
    scaler,
    "../artifacts/scaler.pkl"
)

# Save Label Encoders
joblib.dump(
    label_encoders,
    "../artifacts/label_encoders.pkl"
)

# Save Numerical Imputer
joblib.dump(
    num_imputer,
    "../artifacts/num_imputer.pkl"
)

# Save Categorical Imputer
joblib.dump(
    cat_imputer,
    "../artifacts/cat_imputer.pkl"
)

print("Preprocessing Objects Saved Successfully")

Preprocessing Objects Saved Successfully


In [16]:
# ============================================
# 2.12 Save Feature Columns
# ============================================

feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../artifacts/feature_columns.pkl"
)

print("Feature Columns Saved Successfully")

Feature Columns Saved Successfully


In [17]:
# ============================================
# 2.13 Save Processed Data
# ============================================

# Convert Arrays to DataFrame

X_train_df = pd.DataFrame(
    X_train,
    columns=feature_columns
)

X_test_df = pd.DataFrame(
    X_test,
    columns=feature_columns
)

# Save Processed Train Data
X_train_df.to_csv(
    "../artifacts/X_train_processed.csv",
    index=False
)

# Save Processed Test Data
X_test_df.to_csv(
    "../artifacts/X_test_processed.csv",
    index=False
)

# Save Target Data
y_train.to_csv(
    "../artifacts/y_train.csv",
    index=False
)

y_test.to_csv(
    "../artifacts/y_test.csv",
    index=False
)

print("Processed Data Saved Successfully")

Processed Data Saved Successfully


In [18]:
# ============================================
# 2.14 Final Verification
# ============================================

print("X_train Shape :", X_train.shape)
print("X_test Shape :", X_test.shape)

print("y_train Shape :", y_train.shape)
print("y_test Shape :", y_test.shape)

X_train Shape : (112560, 17)
X_test Shape : (28140, 17)
y_train Shape : (112560,)
y_test Shape : (28140,)


In [19]:
# ============================================
# 2.15 Final Message
# ============================================

print("Data Preprocessing Completed Successfully")

Data Preprocessing Completed Successfully
